# Módulo 5 — Análise de Qualidade via Snowflake

Neste notebook construímos verificações de qualidade que rodam diretamente no Snowflake — aproveitando a escalabilidade do data warehouse para processar grandes volumes.

## O que vamos ver

1. Verificação de nulos por coluna via SQL
2. Análise de chaves duplicadas
3. Regras de consistência em SQL
4. Exportação do relatório de qualidade para Excel

---

> **Nota:** Este notebook usa a classe `SnowflakeConector` definida em `01_conexao_snowflake.ipynb`.
> Se não tiver acesso ao Snowflake, as queries SQL são válidas para estudo e podem ser testadas
> com dados locais simulados (seção alternativa ao final de cada bloco).

In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from datetime import datetime
import re

load_dotenv("../../.env")

# Verificar modo de operação
MODO_SNOWFLAKE = all(
    os.getenv(v) for v in ["SF_ACCOUNT", "SF_USER", "SF_PASSWORD"]
)

if MODO_SNOWFLAKE:
    print("Modo: SNOWFLAKE (credenciais configuradas)")
else:
    print("Modo: LOCAL (sem credenciais Snowflake — usando CSV simulado)")
    print("As queries SQL serão exibidas, mas executadas sobre o CSV local.")

# Carregar dados locais como fallback
df_local = pd.read_csv("../modulo2_dados_com_pandas/dados/consumidores_simulado.csv")
df_local["dat_ultima_leitura"] = pd.to_datetime(df_local["dat_ultima_leitura"], errors="coerce")
print(f"\nDados locais carregados: {len(df_local)} registros")

## 1. Verificação de Nulos por Coluna

In [ ]:
# Query Snowflake para verificar nulos
QUERY_NULOS = """
-- Percentual de nulos por campo no cadastro de UCs
SELECT
    COUNT(*) AS total_registros,

    -- Identificação
    ROUND(SUM(CASE WHEN cpf_cnpj IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
        AS pct_sem_cpf_cnpj,
    ROUND(SUM(CASE WHEN nom_consumidor IS NULL OR TRIM(nom_consumidor) = '' THEN 1 ELSE 0 END)
        / COUNT(*) * 100, 2) AS pct_sem_nome,

    -- Endereço
    ROUND(SUM(CASE WHEN num_cep IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
        AS pct_sem_cep,
    ROUND(SUM(CASE WHEN nom_municipio IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
        AS pct_sem_municipio,

    -- Cadastro técnico
    ROUND(SUM(CASE WHEN num_medidor IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
        AS pct_sem_medidor,
    ROUND(SUM(CASE WHEN dat_ultima_leitura IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
        AS pct_sem_data_leitura,
    ROUND(SUM(CASE WHEN vlr_consumo_medio_kwh IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
        AS pct_sem_consumo,

    -- Geolocalização
    ROUND(SUM(CASE WHEN num_latitude IS NULL OR num_longitude IS NULL THEN 1 ELSE 0 END)
        / COUNT(*) * 100, 2) AS pct_sem_coordenadas

FROM CADASTRO.CONSUMIDORES_UC
WHERE flg_ativo = TRUE;
"""

print("Query SQL para análise de nulos:")
print(QUERY_NULOS)

# Simular resultado com dados locais (equivalente à query acima)
print("\n--- Resultado com dados locais (equivalente) ---")
campos = ["cpf_cnpj", "nom_consumidor", "num_cep", "nom_municipio",
          "num_medidor", "dat_ultima_leitura", "vlr_consumo_medio_kwh"]

resultado_nulos = pd.DataFrame({
    "campo": campos,
    "pct_nulos": [
        round(df_local[c].isnull().sum() / len(df_local) * 100, 2)
        for c in campos
    ]
}).sort_values("pct_nulos", ascending=False)

resultado_nulos["status"] = resultado_nulos["pct_nulos"].apply(
    lambda x: "CRÍTICO" if x > 10 else ("ATENÇÃO" if x > 0 else "OK")
)

print(resultado_nulos.to_string(index=False))

## 2. Análise de Duplicatas

In [ ]:
# Query para detectar duplicatas de chave primária
QUERY_DUPLICATAS = """
-- Detectar duplicatas de cod_consumidor (chave primária)
WITH duplicatas_pk AS (
    SELECT
        cod_consumidor,
        COUNT(*) AS ocorrencias
    FROM CADASTRO.CONSUMIDORES_UC
    GROUP BY cod_consumidor
    HAVING COUNT(*) > 1
),
-- Detectar duplicatas de CPF/CNPJ (mesmo documento, cods diferentes)
duplicatas_doc AS (
    SELECT
        cpf_cnpj,
        COUNT(DISTINCT cod_consumidor) AS qtd_ucs
    FROM CADASTRO.CONSUMIDORES_UC
    WHERE cpf_cnpj IS NOT NULL
    GROUP BY cpf_cnpj
    HAVING COUNT(DISTINCT cod_consumidor) > 1
)
SELECT
    'cod_consumidor' AS tipo_duplicata,
    COUNT(*) AS qtd_duplicatas,
    SUM(ocorrencias) AS total_registros_envolvidos
FROM duplicatas_pk
UNION ALL
SELECT
    'cpf_cnpj',
    COUNT(*),
    SUM(qtd_ucs)
FROM duplicatas_doc;
"""

print("Query SQL para análise de duplicatas:")
print(QUERY_DUPLICATAS)

# Resultado equivalente com dados locais
print("\n--- Resultado com dados locais ---")

dup_cod = df_local[df_local.duplicated("cod_consumidor", keep=False)]
dup_cpf = df_local[
    df_local["cpf_cnpj"].notna() &
    df_local.duplicated("cpf_cnpj", keep=False)
]

resultado_dup = pd.DataFrame([
    {"tipo": "cod_consumidor", "qtd_chaves_duplicadas": df_local["cod_consumidor"].duplicated().sum(),
     "registros_envolvidos": len(dup_cod)},
    {"tipo": "cpf_cnpj", "qtd_chaves_duplicadas": df_local["cpf_cnpj"].dropna().duplicated().sum(),
     "registros_envolvidos": len(dup_cpf)},
])

print(resultado_dup.to_string(index=False))

if len(dup_cod) > 0:
    print(f"\nDetalhes das duplicatas de cod_consumidor:")
    print(dup_cod[["cod_consumidor", "nom_consumidor"]].to_string(index=False))

## 3. Verificações de Consistência

In [ ]:
# Query de consistência tarifária
QUERY_CONSISTENCIA_TARIFARIA = """
-- Verifica inconsistências entre classe de consumo e modalidade tarifária
-- Regra: RESIDENCIAL = B1; RURAL = B2; ILUMINACAO PUBLICA = B4
SELECT
    cod_consumidor,
    nom_consumidor,
    cod_classe_consumo,
    cod_modalidade_tarifaria,
    'MODALIDADE_INCONSISTENTE' AS tipo_problema
FROM CADASTRO.CONSUMIDORES_UC
WHERE
    -- Residencial deve ser sempre B1
    (cod_classe_consumo = 'RESIDENCIAL' AND cod_modalidade_tarifaria <> 'B1')
    OR
    -- Rural deve ser sempre B2
    (cod_classe_consumo = 'RURAL' AND cod_modalidade_tarifaria <> 'B2')
    OR
    -- Iluminação Pública deve ser B4
    (cod_classe_consumo = 'ILUMINACAO PUBLICA' AND cod_modalidade_tarifaria <> 'B4')
ORDER BY cod_classe_consumo, cod_consumidor
LIMIT 100;
"""

# Equivalente local
inconsistentes = df_local[
    ((df_local["cod_classe_consumo"] == "RESIDENCIAL") & (df_local["cod_modalidade_tarifaria"] != "B1")) |
    ((df_local["cod_classe_consumo"] == "RURAL") & (df_local["cod_modalidade_tarifaria"] != "B2")) |
    ((df_local["cod_classe_consumo"] == "ILUMINACAO PUBLICA") & (df_local["cod_modalidade_tarifaria"] != "B4"))
]

print(f"Inconsistências tarifárias encontradas: {len(inconsistentes)}")
if len(inconsistentes) > 0:
    print(inconsistentes[["cod_consumidor", "nom_consumidor", "cod_classe_consumo", "cod_modalidade_tarifaria"]].to_string(index=False))
else:
    print("Nenhuma inconsistência encontrada nos dados locais.")

## 4. Exportar Relatório de Qualidade para Excel

In [ ]:
# Compilar todas as análises e exportar
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
arquivo_saida = f"relatorio_qualidade_{timestamp}.xlsx"

# Preparar dados para o relatório

# 1. Completude
campos_analise = [
    "cod_consumidor", "nom_consumidor", "cpf_cnpj",
    "cod_modalidade_tarifaria", "cod_classe_consumo",
    "nom_municipio", "cod_uf", "num_cep", "num_medidor",
    "flg_ativo", "dat_ultima_leitura", "vlr_consumo_medio_kwh"
]

df_completude = pd.DataFrame([
    {
        "campo": c,
        "total": len(df_local),
        "preenchidos": df_local[c].notna().sum(),
        "nulos": df_local[c].isnull().sum(),
        "pct_preenchido": round(df_local[c].notna().sum() / len(df_local) * 100, 1)
    }
    for c in campos_analise if c in df_local.columns
]).sort_values("pct_preenchido")

df_completude["status"] = df_completude["pct_preenchido"].apply(
    lambda x: "CRÍTICO" if x < 80 else ("ATENÇÃO" if x < 95 else "OK")
)

# 2. Duplicatas
df_duplicatas = df_local[df_local.duplicated("cod_consumidor", keep=False)].copy()

# 3. Resumo executivo
df_resumo = pd.DataFrame([
    {"metrica": "Total de registros", "valor": len(df_local)},
    {"metrica": "Registros ativos", "valor": df_local["flg_ativo"].sum()},
    {"metrica": "Campos com completude < 95%", "valor": (df_completude["pct_preenchido"] < 95).sum()},
    {"metrica": "Campos críticos (< 80%)", "valor": (df_completude["pct_preenchido"] < 80).sum()},
    {"metrica": "Duplicatas de cod_consumidor", "valor": len(df_duplicatas)},
    {"metrica": "Data de geração", "valor": datetime.now().strftime("%d/%m/%Y %H:%M")},
])

# Exportar para Excel
with pd.ExcelWriter(arquivo_saida, engine="openpyxl") as writer:
    df_resumo.to_excel(writer, sheet_name="Resumo", index=False)
    df_completude.to_excel(writer, sheet_name="Completude", index=False)
    if len(df_duplicatas) > 0:
        df_duplicatas.to_excel(writer, sheet_name="Duplicatas", index=False)
    df_local.to_excel(writer, sheet_name="Dados_Completos", index=False)

print(f"Relatório exportado: {arquivo_saida}")
print(f"Abas criadas: Resumo, Completude, {'Duplicatas, ' if len(df_duplicatas) > 0 else ''}Dados_Completos")
print(f"\nResumo executivo:")
print(df_resumo.to_string(index=False))